# Few-shot Hangul Font Generation with Conditional DDPM

Final Colab notebook for the COSE362 Machine Learning project.

This notebook renders Korean font files into 64×64 glyph images, trains a Direct U-Net baseline, trains a Conditional DDPM, generates target-style Hangul glyphs, and evaluates them using L1/MSE/RMSE/SSIM/LPIPS/FID.

## How this notebook should be run from GitHub

When this notebook is opened directly from GitHub in Google Colab, Colab loads only the notebook file. It does not automatically clone the whole repository, so the `src/` folder may not exist in the runtime. Therefore, run the first setup cell after setting `REPO_URL` to the submitted GitHub repository URL.

Expected repository name:

```text
20261R0136COSE362
```

Expected Drive font folder:

```text
/content/drive/MyDrive/ML fonts/fonts/
```

The original font files are not included in the repository because of file size and license considerations. See `docs/dataset.md` for the font list used in our experiment.


In [ ]:
# ============================================================
# Colab / GitHub setup
# ============================================================
# If this notebook is opened directly from GitHub in Colab,
# set REPO_URL to the submitted public repository URL.
# Example:
# REPO_URL = "https://github.com/Github-ID/20261R0136COSE362.git"

REPO_URL = "https://github.com/hongjw712/20261R0136COSE362.git"
PROJECT_DIR = "/content/20261R0136COSE362"

from pathlib import Path
import os
import subprocess
import sys

project_dir = Path(PROJECT_DIR)
src_file = project_dir / "src" / "hangul_font_diffusion_mvp.py"

# Clone the full repository only when src/ is not already available.
# If REPO_URL still contains the placeholder, print clear instructions
# instead of failing silently.
if not src_file.exists():
    if "<Github-ID>" in REPO_URL:
        print("[Setup notice]")
        print("The repository source file was not found in /content/20261R0136COSE362/src/.")
        print("If you opened this notebook from GitHub, replace <Github-ID> in REPO_URL")
        print("with the submitted GitHub ID, then rerun this cell.")
        print("Example: https://github.com/Github-ID/20261R0136COSE362.git")
    else:
        if project_dir.exists():
            subprocess.run(["rm", "-rf", str(project_dir)], check=True)
        subprocess.run(["git", "clone", REPO_URL, str(project_dir)], check=True)

# Use the cloned repository as the working directory when available.
if project_dir.exists():
    os.chdir(project_dir)
    print("Working directory:", Path.cwd())

# System fonts and Python packages
subprocess.run("apt-get -qq update", shell=True, check=False)
subprocess.run("apt-get -qq install -y fonts-noto-cjk fonts-nanum", shell=True, check=False)

req = Path("requirements-colab.txt")
if req.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow", "tqdm", "matplotlib", "pandas", "scipy", "lpips"], check=False)

print("Setup cell finished.")


The next cell mounts Google Drive and imports the project source code. If the source import fails, rerun the first setup cell after setting `REPO_URL` correctly, or verify that `src/hangul_font_diffusion_mvp.py` exists in the cloned repository.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import importlib
from pathlib import Path

# Google Drive project root used for font files and generated outputs
ROOT = "/content/drive/MyDrive/ML fonts"
drive_src = Path(ROOT) / "src"

# GitHub/Colab fallback paths. These allow the submitted repository to run
# even when Drive does not contain a separate src folder.
repo_src_candidates = [
    Path("/content/20261R0136COSE362/src"),
    Path.cwd() / "src",
    Path.cwd().parent / "src",
]

src_candidates = [drive_src] + repo_src_candidates
SRC = None
for cand in src_candidates:
    if (cand / "hangul_font_diffusion_mvp.py").exists():
        SRC = cand
        break

if SRC is None:
    raise FileNotFoundError(
        "Could not find hangul_font_diffusion_mvp.py. "
        "Clone the GitHub repository to /content/20261R0136COSE362 "
        "or place the file under /content/drive/MyDrive/ML fonts/src/."
    )

print("ROOT:", ROOT)
print("Using SRC:", SRC)
print("Files in SRC:", sorted(p.name for p in SRC.iterdir()))

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

importlib.invalidate_caches()

import hangul_font_diffusion_mvp as hfdm
from hangul_font_diffusion_mvp import *

DEFAULT_REFERENCE_CHARS = [
    "가", "넋", "둘", "레", "묽", "빛", "산", "잊",
    "쾌", "흙"
]

# Keep the module-level reference characters synchronized.
hfdm.DEFAULT_REFERENCE_CHARS = DEFAULT_REFERENCE_CHARS

set_seed(42)

print("Import success!")
print("Reference chars:", "".join(DEFAULT_REFERENCE_CHARS))


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## Render a compact 64x64 dataset

This notebook renders a compact but non-trivial dataset for Colab execution. The default `max_chars=256` keeps the run time manageable while still including the configured reference/target characters used for qualitative comparison. Increase `max_chars` later only if the baseline and DDPM cells already run successfully.


In [ ]:
import shutil
from PIL import Image
from pathlib import Path

work = Path('/content/hangul_font_mvp')
render_dir = work / 'rendered'
out_dir = work / 'outputs'

# 이전 실행의 폰트가 섞이지 않도록 매번 초기화
font_dir = work / 'fonts'
if font_dir.exists():
    shutil.rmtree(font_dir)
font_dir.mkdir(parents=True, exist_ok=True)

# ── 여기만 수정하세요 ────────────────────────────────────────────────────────
MY_FONT_DIR = '/content/drive/MyDrive/ML fonts/fonts'  # Drive에 .ttf/.otf/.ttc 파일을 넣어둔 폴더 경로
# ────────────────────────────────────────────────────────────────────────────

# 1) 시스템 NanumGothic을 base font로 자동 추가 (글자 구조 참조용)
system_nanum = next(Path('/usr/share/fonts').rglob('NanumGothic.ttf'), None)
if system_nanum:
    shutil.copy2(system_nanum, font_dir / system_nanum.name)
    print('Base font (system):', system_nanum.name)
else:
    print('경고: NanumGothic.ttf를 찾지 못했습니다. Cell 1을 먼저 실행하세요.')

# 2) Drive의 내 폰트 복사 (스타일 학습 대상)
my_font_dir = Path(MY_FONT_DIR)
my_fonts = sorted(
    p for p in my_font_dir.iterdir()
    if p.suffix.lower() in {'.ttf', '.otf', '.ttc'}
) if my_font_dir.exists() else []

if my_fonts:
    for p in my_fonts:
        shutil.copy2(p, font_dir / p.name)
else:
    print(f'경고: {MY_FONT_DIR} 에서 폰트를 찾지 못했습니다.')

print('Fonts copied:', sorted(p.name for p in font_dir.iterdir()))

# Compact Colab setting: increasing max_chars improves coverage but increases runtime.
records = render_font_dataset(font_dir, render_dir, glyph_size=64, max_chars=256, seed=42)
print('Rendered records:', len(records))
print('Metadata:', render_dir / 'metadata.jsonl')


In [ ]:
from IPython.display import display
from pathlib import Path

metadata_path = render_dir / 'metadata.jsonl'
loaded = load_records(metadata_path)
font_ids = sorted({r['font_id'] for r in loaded})

# font_id가 아니라 실제 폰트 파일명을 출력하기 위한 매핑
font_file_by_id = {}
for r in loaded:
    fid = str(r['font_id'])
    fp = Path(str(r.get('font_path', '')))
    font_file_by_id.setdefault(fid, fp.name if fp.name else fid)

print('fonts used in rendered dataset:')
for i, fid in enumerate(font_ids, 1):
    print(f'{i:02d}. {font_file_by_id.get(fid, fid)}')

sample_imgs = [Image.open(r['image_path']) for r in loaded[:24]]
# PIL 기본 폰트가 한글 라벨을 못 그려 □로 깨질 수 있으므로 라벨 없이 표시
# 실제 글자 목록은 metadata 파일과 아래 출력 로그에서 확인할 수 있습니다.
display(make_image_grid(sample_imgs, labels=None, cols=8))


자모 임베딩 테스트

In [ ]:
from collections import Counter

CHO_LIST = ["ㄱ", "ㄲ", "ㄴ", "ㄷ", "ㄸ", "ㄹ", "ㅁ", "ㅂ", "ㅃ", "ㅅ", "ㅆ", "ㅇ", "ㅈ", "ㅉ", "ㅊ", "ㅋ", "ㅌ", "ㅍ", "ㅎ"]
JUNG_LIST = ["ㅏ", "ㅐ", "ㅑ", "ㅒ", "ㅓ", "ㅔ", "ㅕ", "ㅖ", "ㅗ", "ㅘ", "ㅙ", "ㅚ", "ㅛ", "ㅜ", "ㅝ", "ㅞ", "ㅟ", "ㅠ", "ㅡ", "ㅢ", "ㅣ"]
JONG_LIST = ["", "ㄱ", "ㄲ", "ㄳ", "ㄴ", "ㄵ", "ㄶ", "ㄷ", "ㄹ", "ㄺ", "ㄻ", "ㄼ", "ㄽ", "ㄾ", "ㄿ", "ㅀ", "ㅁ", "ㅂ", "ㅄ", "ㅅ", "ㅆ", "ㅇ", "ㅈ", "ㅊ", "ㅋ", "ㅌ", "ㅍ", "ㅎ"]

def decompose_hangul_char(ch):
    code = ord(ch) - ord("가")
    if code < 0 or code >= 11172:
        return None
    cho = code // (21 * 28)
    jung = (code % (21 * 28)) // 28
    jong = code % 28
    return cho, jung, jong

records = load_records(metadata_path)
chars = sorted({r["char"] for r in records})

cho_counter = Counter()
jung_counter = Counter()
jong_counter = Counter()

for ch in chars:
    dec = decompose_hangul_char(ch)
    if dec is None:
        continue
    cho, jung, jong = dec
    cho_counter[CHO_LIST[cho]] += 1
    jung_counter[JUNG_LIST[jung]] += 1
    jong_counter[JONG_LIST[jong]] += 1

print("총 unique chars:", len(chars))

print("\n[초성 분포]")
for j in CHO_LIST:
    print(j, cho_counter[j])

print("\n[중성 분포]")
for j in JUNG_LIST:
    print(j, jung_counter[j])

print("\n[종성 분포]")
for j in JONG_LIST:
    label = "없음" if j == "" else j
    print(label, jong_counter[j])

## Direct U-Net baseline

This gives a quick quality floor before running diffusion.

In [ ]:
# @title
baseline_cfg = TrainConfig(
    metadata_path=str(metadata_path),
    output_dir=str(out_dir),
    epochs=999,          # 실제 종료는 max_steps 기준
    batch_size=32,
    max_steps=8000,      # 1200은 너무 짧아서 글자 윤곽만 흐리게 나옵니다.
    base_channels=32,    # 64x64 흑백 글리프용 경량 U-Net. OOM이면 24로 낮추세요.
    reference_count=len(DEFAULT_REFERENCE_CHARS),
    reference_chars=tuple(DEFAULT_REFERENCE_CHARS),
    diffusion_steps=200,
    beta_end=0.02,
    lr=2e-4,
    num_workers=2,
    recon_weight=0.0,
)
baseline_model, baseline_history, baseline_ckpt = train_unet_baseline(baseline_cfg)
baseline_ckpt, baseline_history[-1]


## Conditional DDPM training

This cell trains the Conditional DDPM using the project configuration. The runtime depends on GPU availability and the number of rendered fonts/characters.

In [ ]:
ddpm_cfg = TrainConfig(
    metadata_path=str(metadata_path),
    output_dir=str(out_dir),
    epochs=999,          # 실제 종료는 max_steps 기준
    batch_size=32,
    max_steps=20000,     # 2500은 DDPM이 거의 배경만 내므로 최소 20000 권장
    base_channels=120,    # OOM이면 24로 낮추세요.
    reference_count=len(DEFAULT_REFERENCE_CHARS),
    reference_chars=tuple(DEFAULT_REFERENCE_CHARS),
    diffusion_steps=300, # Number of diffusion steps used by the DDPM.
    beta_end=0.02,
    lr=1e-4,
    num_workers=2,
    recon_weight=0.1,    # Auxiliary reconstruction loss weight.
)
ddpm_model, ddpm_history, ddpm_ckpt = train_conditional_ddpm(ddpm_cfg)
ddpm_ckpt, ddpm_history[-1]


## Evaluate one few-shot batch

In [ ]:
# @title
target_font = font_ids[-1]
reference_chars = [c for c in DEFAULT_REFERENCE_CHARS if c in index_records(loaded)[target_font]]
missing_reference_chars = [c for c in DEFAULT_REFERENCE_CHARS if c not in index_records(loaded)[target_font]]
if missing_reference_chars:
    print('warning: target font에서 렌더링되지 않은 reference chars:', ''.join(missing_reference_chars))
target_char = '값' if '값' in index_records(loaded)[target_font] else loaded[0]['char']
batch = build_fewshot_batch(target_font, target_char, reference_chars, metadata_path=metadata_path, reference_count=len(DEFAULT_REFERENCE_CHARS))
print('baseline metrics:', evaluate_checkpoint_on_batch(baseline_ckpt, batch))
print('ddpm metrics:', evaluate_checkpoint_on_batch(ddpm_ckpt, batch))

In [ ]:
# =========================================
# Optional single-batch evaluation
# The baseline checkpoint is optional; DDPM evaluation still runs if available.
# =========================================

from pathlib import Path
import os

def _get_global(names, default=None):
    """Return the first available global variable from the candidate names."""
    for name in names:
        if name in globals() and globals()[name] is not None:
            return globals()[name]
    return default

def _checkpoint_available(ckpt):
    """Check whether a checkpoint object or checkpoint path is available."""
    if ckpt is None:
        return False
    if isinstance(ckpt, (str, os.PathLike)):
        return Path(ckpt).exists()
    return True

def _safe_evaluate_checkpoint(label, ckpt, batch):
    """Evaluate a checkpoint when it is available."""
    if not _checkpoint_available(ckpt):
        print(f"{label} metrics: skipped (checkpoint not found / not trained)")
        return None

    try:
        metrics = evaluate_checkpoint_on_batch(ckpt, batch)
        print(f"{label} metrics:", metrics)
        return metrics
    except Exception as e:
        print(f"{label} metrics: failed ({type(e).__name__}: {e})")
        return None


# Build the font/character index once.
records_by_font = index_records(loaded)

# Select the target font for quick evaluation.
target_font = font_ids[-1]
target_font_records = records_by_font[target_font]

# Use only rendered reference characters.
reference_chars = [
    c for c in DEFAULT_REFERENCE_CHARS
    if c in target_font_records
]

missing_reference_chars = [
    c for c in DEFAULT_REFERENCE_CHARS
    if c not in target_font_records
]

if missing_reference_chars:
    print(
        "warning: target font에서 렌더링되지 않은 reference chars:",
        "".join(missing_reference_chars)
    )

if len(reference_chars) == 0:
    raise ValueError(
        "사용 가능한 reference_chars가 없습니다. "
        "DEFAULT_REFERENCE_CHARS 또는 렌더링 데이터를 확인하세요."
    )

print("target_font:", target_font)
print("reference_chars:", "".join(reference_chars))

# Select one target character for quick evaluation.
if "값" in target_font_records:
    target_char = "값"
else:
    # Fall back to the first available character.
    target_char = next(iter(target_font_records.keys()))

print("target_char:", target_char)

# few-shot batch 생성
# reference_count는 실제 사용 가능한 reference_chars 개수로 맞춤
batch = build_fewshot_batch(
    target_font,
    target_char,
    reference_chars,
    metadata_path=metadata_path,
    reference_count=len(reference_chars)
)

# checkpoint 변수 가져오기
baseline_ckpt_safe = _get_global(["baseline_ckpt"], None)
ddpm_ckpt_safe = _get_global(
    ["ddpm_ckpt", "conditional_ddpm_ckpt", "cond_ddpm_ckpt"],
    None
)

# baseline은 없으면 자동 skip
baseline_metrics = _safe_evaluate_checkpoint(
    "baseline",
    baseline_ckpt_safe,
    batch
)

# DDPM은 있으면 평가
ddpm_metrics = _safe_evaluate_checkpoint(
    "ddpm",
    ddpm_ckpt_safe,
    batch
)

## Quantitative metric evaluation

이 셀은 평가 target 글자를 기본 128개로 늘려 L1 / MSE / RMSE / SSIM / LPIPS / FID를 계산합니다.  
실행 위치: `Direct U-Net baseline`과 `Conditional DDPM` 학습 및 checkpoint 생성이 끝난 뒤 실행하세요.  
Baseline checkpoint가 없으면 baseline은 자동으로 skip하고 DDPM만 평가합니다.


In [ ]:
# =========================================
# Quantitative evaluation metrics
# L1 / MSE / RMSE / SSIM / LPIPS / FID
# Uses more target characters than the quick qualitative example
# =========================================

from pathlib import Path
from PIL import Image
from datetime import datetime
import os
import sys
import subprocess
import json
import random
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

try:
    from IPython.display import display, Markdown
except Exception:
    display = None
    Markdown = None

# ─────────────────────────────────────────────────────────────────────────────
# 0. Evaluation setting
# ─────────────────────────────────────────────────────────────────────────────
# The default evaluates 128 target characters when the rendered dataset supports it.
EVAL_TARGET_COUNT = 128
EVAL_RANDOM_SEED = 42
FID_BATCH_SIZE = 32

# FID uses Inception features and may download torchvision weights on first run.
ENABLE_LPIPS = True
ENABLE_FID = True

# ─────────────────────────────────────────────────────────────────────────────
# 1. Utility helpers
# ─────────────────────────────────────────────────────────────────────────────
def has_var(name):
    return name in globals() and globals()[name] is not None

def get_var(name, default=None):
    return globals()[name] if has_var(name) else default

def get_first_existing(names, default=None):
    for name in names:
        if has_var(name):
            return globals()[name]
    return default

def ckpt_exists(ckpt):
    if ckpt is None:
        return False
    if isinstance(ckpt, (str, os.PathLike, Path)):
        return Path(ckpt).exists()
    return True

def record_to_path(record):
    if isinstance(record, (str, os.PathLike, Path)):
        return Path(record)
    candidate_keys = ["image_path", "path", "file_path", "png_path", "img_path", "glyph_path"]
    if isinstance(record, dict):
        for k in candidate_keys:
            if k in record:
                return Path(record[k])
    raise KeyError(f"이미지 경로 key를 찾을 수 없음. record={record}")

def pil_to_tensor_gray(img, image_size=None):
    if isinstance(img, (str, os.PathLike, Path)):
        img = Image.open(img)
    img = img.convert("L")
    if image_size is not None:
        img = img.resize((image_size, image_size), Image.BILINEAR)
    arr = np.array(img).astype(np.float32) / 255.0
    return torch.from_numpy(arr).unsqueeze(0)  # [1,H,W]

def make_gt_batch(gt_paths, image_size=None):
    return torch.stack([pil_to_tensor_gray(p, image_size=image_size) for p in gt_paths], dim=0)

def make_gen_batch(images, image_size=None):
    return torch.stack([pil_to_tensor_gray(img, image_size=image_size) for img in images], dim=0)

# ─────────────────────────────────────────────────────────────────────────────
# 2. SSIM implementation
# ─────────────────────────────────────────────────────────────────────────────
def ssim_per_image(x, y, window_size=11):
    """
    x, y: [B,1,H,W], range [0,1]
    return: [B]
    """
    assert x.shape == y.shape, (x.shape, y.shape)
    device = x.device
    channel = x.size(1)
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    window = torch.ones(channel, 1, window_size, window_size, device=device, dtype=x.dtype)
    window = window / (window_size * window_size)
    padding = window_size // 2

    mu_x = F.conv2d(x, window, padding=padding, groups=channel)
    mu_y = F.conv2d(y, window, padding=padding, groups=channel)
    mu_x_sq = mu_x ** 2
    mu_y_sq = mu_y ** 2
    mu_xy = mu_x * mu_y

    sigma_x_sq = F.conv2d(x * x, window, padding=padding, groups=channel) - mu_x_sq
    sigma_y_sq = F.conv2d(y * y, window, padding=padding, groups=channel) - mu_y_sq
    sigma_xy = F.conv2d(x * y, window, padding=padding, groups=channel) - mu_xy

    ssim_map = ((2 * mu_xy + C1) * (2 * sigma_xy + C2)) / (
        (mu_x_sq + mu_y_sq + C1) * (sigma_x_sq + sigma_y_sq + C2)
    )
    return ssim_map.mean(dim=[1, 2, 3])

# ─────────────────────────────────────────────────────────────────────────────
# 3. LPIPS
# ─────────────────────────────────────────────────────────────────────────────
def load_lpips_model(device):
    if not ENABLE_LPIPS:
        return None
    try:
        import lpips
    except Exception:
        print("[info] lpips package not found. Installing lpips...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lpips"])
            import lpips
        except Exception as e:
            print("[warning] LPIPS 설치 실패. LPIPS는 skip됩니다:", e)
            return None
    try:
        model = lpips.LPIPS(net="alex").to(device)
        model.eval()
        return model
    except Exception as e:
        print("[warning] LPIPS model load 실패. LPIPS는 skip됩니다:", e)
        return None

def lpips_per_image(x, y, lpips_model, device):
    if lpips_model is None:
        return None
    x3 = x.repeat(1, 3, 1, 1).to(device)
    y3 = y.repeat(1, 3, 1, 1).to(device)
    x3 = x3 * 2.0 - 1.0
    y3 = y3 * 2.0 - 1.0
    with torch.no_grad():
        vals = lpips_model(x3, y3)
    return vals.view(-1).detach().cpu()

# ─────────────────────────────────────────────────────────────────────────────
# 4. FID implementation
# ─────────────────────────────────────────────────────────────────────────────
def _sqrtm_psd(mat):
    """Matrix square root for covariance product. scipy 우선, 실패하면 eigen fallback."""
    try:
        from scipy import linalg
        out = linalg.sqrtm(mat)
        if np.iscomplexobj(out):
            out = out.real
        return out
    except Exception as e:
        print("[warning] scipy sqrtm failed, using eigen fallback:", e)
        mat = (mat + mat.T) / 2.0
        vals, vecs = np.linalg.eigh(mat)
        vals = np.clip(vals, 0, None)
        return (vecs * np.sqrt(vals)).dot(vecs.T)

def fid_from_features(feats1, feats2, eps=1e-6):
    """
    feats1, feats2: numpy [N,D]
    lower is better
    """
    feats1 = np.asarray(feats1, dtype=np.float64)
    feats2 = np.asarray(feats2, dtype=np.float64)
    if feats1.shape[0] < 2 or feats2.shape[0] < 2:
        return None

    mu1, mu2 = feats1.mean(axis=0), feats2.mean(axis=0)
    sigma1 = np.cov(feats1, rowvar=False)
    sigma2 = np.cov(feats2, rowvar=False)

    # numerical stability
    if not np.isfinite(sigma1).all():
        sigma1 = np.nan_to_num(sigma1)
    if not np.isfinite(sigma2).all():
        sigma2 = np.nan_to_num(sigma2)

    covmean = _sqrtm_psd(sigma1.dot(sigma2))
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = _sqrtm_psd((sigma1 + offset).dot(sigma2 + offset))

    diff = mu1 - mu2
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2.0 * covmean)
    return float(np.real(fid))

def load_inception_feature_model(device):
    if not ENABLE_FID:
        return None
    try:
        import torchvision
        from torchvision.models import inception_v3, Inception_V3_Weights
        weights = Inception_V3_Weights.DEFAULT
        model = inception_v3(weights=weights, aux_logits=True, transform_input=False)
        model.fc = torch.nn.Identity()
        model.eval().to(device)
        return model
    except Exception as e:
        print("[warning] Inception model load 실패. FID는 skip됩니다:", e)
        return None

def inception_features_from_tensor(x, model, device, batch_size=32):
    """
    x: [N,1,H,W], range [0,1]
    return: numpy [N,2048]
    """
    if model is None:
        return None
    feats = []
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    with torch.no_grad():
        for i in range(0, x.size(0), batch_size):
            batch = x[i:i + batch_size].to(device)
            batch = batch.repeat(1, 3, 1, 1)
            batch = F.interpolate(batch, size=(299, 299), mode="bilinear", align_corners=False)
            batch = (batch - mean) / std
            out = model(batch)
            if isinstance(out, tuple):
                out = out[0]
            feats.append(out.detach().cpu().numpy())
    return np.concatenate(feats, axis=0)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Build large evaluation target set
# ─────────────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

records_eval = load_records(metadata_path)
indexed_eval = index_records(records_eval)
all_font_ids_eval = sorted(indexed_eval)
base_font_id_eval = get_var("base_font_id", all_font_ids_eval[0])

# target_font는 이전 summary 셀에서 정한 값이 있으면 재사용. 없으면 DDPM test split 우선.
if has_var("target_font"):
    target_font_eval = target_font
else:
    try:
        train_fonts_eval, test_fonts_eval = split_fonts_by_id(
            records_eval,
            test_font_ratio=ddpm_cfg.test_font_ratio,
            holdout_fonts=ddpm_cfg.holdout_fonts,
            base_font_id=base_font_id_eval,
            seed=ddpm_cfg.seed,
        )
        test_target_fonts_eval = [f for f in test_fonts_eval if f != base_font_id_eval]
        target_font_eval = test_target_fonts_eval[0] if test_target_fonts_eval else all_font_ids_eval[-1]
    except Exception:
        target_font_eval = all_font_ids_eval[-1]

# reference chars는 기존 DEFAULT_REFERENCE_CHARS 중 실제 target font에 있는 것만 사용.
reference_chars_eval = [
    c for c in DEFAULT_REFERENCE_CHARS
    if c in indexed_eval.get(target_font_eval, {})
]
if not reference_chars_eval:
    raise ValueError("reference_chars_eval이 비어 있습니다. DEFAULT_REFERENCE_CHARS 또는 target font 렌더링 상태를 확인하세요.")

reference_paths_eval = pick_reference_image_paths(
    metadata_path,
    target_font_eval,
    reference_chars_eval,
)

# base font와 target font 둘 다 존재하고 reference가 아닌 글자를 후보로 사용.
all_candidate_chars = [
    c for c in indexed_eval[base_font_id_eval].keys()
    if c in indexed_eval.get(target_font_eval, {}) and c not in reference_chars_eval
]
all_candidate_chars = sorted(set(all_candidate_chars))

if len(all_candidate_chars) < 100:
    print(f"[warning] 평가 후보 글자가 {len(all_candidate_chars)}개뿐입니다. 논문 비교용으로는 최소 100개 이상 권장합니다.")

rng = random.Random(EVAL_RANDOM_SEED)
# 기존 DEFAULT_TARGET_CHARS가 있으면 먼저 포함하고, 나머지를 랜덤으로 채움.
priority_chars = []
if "DEFAULT_TARGET_CHARS" in globals():
    priority_chars = [
        c for c in DEFAULT_TARGET_CHARS
        if c in all_candidate_chars
    ]

remaining_chars = [c for c in all_candidate_chars if c not in priority_chars]
rng.shuffle(remaining_chars)

eval_target_chars = (priority_chars + remaining_chars)[:min(EVAL_TARGET_COUNT, len(all_candidate_chars))]

print("=== Large evaluation setup ===")
print("target_font:", target_font_eval)
print("base_font:", base_font_id_eval)
print("reference_chars:", "".join(reference_chars_eval), f"({len(reference_chars_eval)} chars)")
print("candidate target chars:", len(all_candidate_chars))
print("selected eval target chars:", len(eval_target_chars))
print("first 50 eval target chars:", "".join(eval_target_chars[:50]))
print("================================")

if len(eval_target_chars) < 2:
    raise ValueError("FID/metric 평가를 위한 target char 수가 너무 적습니다.")

def infer_image_size(default=64):
    for obj_name in ["ddpm_cfg", "baseline_cfg"]:
        obj = get_var(obj_name, None)
        if obj is not None and hasattr(obj, "image_size"):
            return int(obj.image_size)
    return default

image_size_eval = infer_image_size(64)
gt_paths_eval = [record_to_path(indexed_eval[target_font_eval][ch]) for ch in eval_target_chars]

# ─────────────────────────────────────────────────────────────────────────────
# 6. Generate images for selected target chars
# ─────────────────────────────────────────────────────────────────────────────
# 저장 위치 설정
if has_var("summary_dir"):
    metric_root = Path(summary_dir)
else:
    drive_root = Path("/content/drive/MyDrive/ML fonts/team_share_results")
    metric_root = drive_root / f"metric_eval_final_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    metric_root.mkdir(parents=True, exist_ok=True)

metric_run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
metric_dir = metric_root / f"large_metric_eval_{metric_run_id}"
metric_dir.mkdir(parents=True, exist_ok=True)

baseline_ckpt_eval = get_var("baseline_ckpt", None)
ddpm_ckpt_eval = get_first_existing(["ddpm_ckpt", "conditional_ddpm_ckpt", "cond_ddpm_ckpt"], None)

def generate_images_for_metrics(model_key, ckpt):
    if not ckpt_exists(ckpt):
        print(f"[info] {model_key}: checkpoint 없음 → skip")
        return None, None

    out_dir = metric_dir / f"{model_key}_generated_{metric_run_id}"
    out_dir.mkdir(parents=True, exist_ok=True)

    random.seed(EVAL_RANDOM_SEED)
    np.random.seed(EVAL_RANDOM_SEED)
    torch.manual_seed(EVAL_RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(EVAL_RANDOM_SEED)

    imgs = generate_glyphs(
        ckpt,
        reference_paths_eval,
        eval_target_chars,
        metadata_path=metadata_path,
        base_font_id=base_font_id_eval,
        out_dir=out_dir,
    )
    print(f"[info] {model_key}: generated {len(imgs)} images → {out_dir}")
    return imgs, out_dir

baseline_images_eval, baseline_gen_dir = generate_images_for_metrics("baseline", baseline_ckpt_eval)
ddpm_images_eval, ddpm_gen_dir = generate_images_for_metrics("ddpm", ddpm_ckpt_eval)

# ─────────────────────────────────────────────────────────────────────────────
# 7. Compute L1 / MSE / RMSE / SSIM / LPIPS / FID
# ─────────────────────────────────────────────────────────────────────────────
lpips_model = load_lpips_model(device)
inception_model = load_inception_feature_model(device)

gt_tensor = make_gt_batch(gt_paths_eval, image_size=image_size_eval).to(device)

def evaluate_image_set(model_name, generated_images):
    if generated_images is None:
        return [], None

    gen_tensor = make_gen_batch(generated_images, image_size=image_size_eval).to(device)

    with torch.no_grad():
        l1_vals = torch.mean(torch.abs(gen_tensor - gt_tensor), dim=[1, 2, 3]).detach().cpu()
        mse_vals = torch.mean((gen_tensor - gt_tensor) ** 2, dim=[1, 2, 3]).detach().cpu()
        rmse_vals = torch.sqrt(mse_vals)
        ssim_vals = ssim_per_image(gen_tensor, gt_tensor).detach().cpu()

    lpips_vals = lpips_per_image(gen_tensor.detach().cpu(), gt_tensor.detach().cpu(), lpips_model, device)

    fid_value = None
    if inception_model is not None:
        try:
            gt_feats = inception_features_from_tensor(gt_tensor.detach().cpu(), inception_model, device, batch_size=FID_BATCH_SIZE)
            gen_feats = inception_features_from_tensor(gen_tensor.detach().cpu(), inception_model, device, batch_size=FID_BATCH_SIZE)
            fid_value = fid_from_features(gt_feats, gen_feats)
        except Exception as e:
            print(f"[warning] {model_name} FID 계산 실패:", e)
            fid_value = None

    rows = []
    for i, ch in enumerate(eval_target_chars):
        rows.append({
            "model": model_name,
            "char": ch,
            "l1": float(l1_vals[i]),
            "mse": float(mse_vals[i]),
            "rmse": float(rmse_vals[i]),
            "ssim": float(ssim_vals[i]),
            "lpips": float(lpips_vals[i]) if lpips_vals is not None else None,
            "fid": fid_value,  # FID는 set-level metric이므로 모든 row에 같은 값 저장
        })
    return rows, fid_value

metric_rows = []
fid_rows = []

rows, fid_val = evaluate_image_set("Direct U-Net baseline", baseline_images_eval)
metric_rows.extend(rows)
if rows:
    fid_rows.append({"model": "Direct U-Net baseline", "fid": fid_val})

rows, fid_val = evaluate_image_set("Conditional DDPM", ddpm_images_eval)
metric_rows.extend(rows)
if rows:
    fid_rows.append({"model": "Conditional DDPM", "fid": fid_val})

metrics_df = pd.DataFrame(metric_rows)
if metrics_df.empty:
    raise RuntimeError("평가할 생성 이미지가 없습니다. checkpoint 또는 generate_glyphs 결과를 확인하세요.")

summary_df = (
    metrics_df
    .groupby("model", dropna=False)
    .agg({
        "l1": "mean",
        "mse": "mean",
        "rmse": "mean",
        "ssim": "mean",
        "lpips": "mean",
    })
    .reset_index()
)

fid_df = pd.DataFrame(fid_rows)
if not fid_df.empty:
    summary_df = summary_df.merge(fid_df, on="model", how="left")
else:
    summary_df["fid"] = None

print("\n=== Final mean metrics by model ===")
if display:
    display(summary_df)
else:
    print(summary_df)

print("\n=== Final per-character metrics sample ===")
if display:
    display(metrics_df.head(20))
else:
    print(metrics_df.head(20))

# ─────────────────────────────────────────────────────────────────────────────
# 8. Save metric files
# ─────────────────────────────────────────────────────────────────────────────
metrics_csv_path = metric_dir / f"generation_metrics_per_char_final_{metric_run_id}.csv"
summary_csv_path = metric_dir / f"generation_metrics_summary_final_{metric_run_id}.csv"
metrics_json_path = metric_dir / f"generation_metrics_final_{metric_run_id}.json"
metrics_md_path = metric_dir / f"generation_metrics_summary_final_{metric_run_id}.md"

metrics_df.to_csv(metrics_csv_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

save_obj = {
    "eval_target_count_requested": EVAL_TARGET_COUNT,
    "eval_target_count_actual": len(eval_target_chars),
    "target_font": str(target_font_eval),
    "base_font": str(base_font_id_eval),
    "reference_chars": list(reference_chars_eval),
    "target_chars": list(eval_target_chars),
    "image_size": image_size_eval,
    "metric_notes": {
        "l1": "lower is better",
        "mse": "lower is better",
        "rmse": "lower is better",
        "ssim": "higher is better",
        "lpips": "lower is better",
        "fid": "lower is better; set-level metric computed from Inception features",
    },
    "summary": summary_df.to_dict(orient="records"),
    "per_char": metrics_df.to_dict(orient="records"),
}

with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(save_obj, f, ensure_ascii=False, indent=2)

md_text = "# Final Generation Metrics Summary\n\n"
md_text += f"- target font: `{target_font_eval}`\n"
md_text += f"- base/content font: `{base_font_id_eval}`\n"
md_text += f"- reference chars: `{''.join(reference_chars_eval)}`\n"
md_text += f"- eval target count requested: `{EVAL_TARGET_COUNT}`\n"
md_text += f"- eval target count actual: `{len(eval_target_chars)}`\n"
md_text += f"- image size: `{image_size_eval}`\n"
md_text += f"- metric output dir: `{metric_dir}`\n\n"
md_text += "## Mean metrics by model\n\n"
md_text += summary_df.to_markdown(index=False)
md_text += "\n\n## Per-character metrics\n\n"
md_text += metrics_df.to_markdown(index=False)
md_text += "\n\n"
md_text += "## Notes\n\n"
md_text += "- L1 / MSE / RMSE / LPIPS / FID는 낮을수록 좋습니다.\n"
md_text += "- SSIM은 높을수록 좋습니다.\n"
md_text += "- FID는 target 글자 집합 전체에 대한 set-level metric이므로 per-character 값이 아니라 모델별 하나의 값입니다.\n"
md_text += "- 현재 데이터가 폰트당 256자라면, 이 평가는 전체 한글 11,172자 평가가 아니라 256자 렌더링 subset 내부의 large-sample 평가입니다.\n"

metrics_md_path.write_text(md_text, encoding="utf-8")

if display and Markdown:
    display(Markdown(md_text))

print("\nSaved final metric files:")
print("metric dir  :", metric_dir)
print("per-char csv:", metrics_csv_path)
print("summary csv :", summary_csv_path)
print("json        :", metrics_json_path)
print("markdown    :", metrics_md_path)
